<a href="https://colab.research.google.com/github/HARSITHRAM/Interactive-Online-Class/blob/main/Int_class.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install paho-mqtt deepface
!wget https://github.com/googlefonts/roboto/raw/main/src/hinted/Roboto-Bold.ttf -O /content/bold.ttf
!apt-get update
!apt-get install -y libfreetype6-dev libpng-dev
!pip install -U Pillow

In [4]:
from google.colab.patches import cv2_imshow
from IPython.display import display, clear_output, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import cv2
import numpy as np
from deepface import DeepFace
from PIL import Image
import time
import paho.mqtt.client as mqtt
import socket
import re
from collections import deque

# Function to convert JS video stream to OpenCV image
def js_to_image(js_reply):
    image_bytes = b64decode(js_reply.split(',')[1])
    jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
    img = cv2.imdecode(jpg_as_np, flags=1)
    return img

# Overlay emotion on the frame
def overlay_emotion_text(frame, emotion, position=(30, 30), font_scale=1, thickness=2):
    cv2.putText(frame, emotion, position, cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 255, 0), thickness, cv2.LINE_AA)

# Get video frame from webcam
def video_frame():
    js_code = '''
        async function capture() {
            const video = document.createElement('video');
            const canvas = document.createElement('canvas');
            const ctx = canvas.getContext('2d');
            const stream = await navigator.mediaDevices.getUserMedia({ video: true });
            video.srcObject = stream;
            await video.play();
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            ctx.drawImage(video, 0, 0, canvas.width, canvas.height);
            const img = canvas.toDataURL('image/jpeg');
            stream.getTracks().forEach(track => track.stop());
            return img;
        }
        capture();
    '''
    return eval_js(js_code)

# MQTT setup
broker = "b7567dff6628421e8bfca997634b396b.s1.eu.hivemq.cloud"
port = 8883
topic = "student_status"

client = mqtt.Client(client_id="colab_emotion", protocol=mqtt.MQTTv311)
client.tls_set()
client.username_pw_set("HARSITHRAM", "Ram@2005")

try:
    client.connect(broker, port)
    print("Connected to MQTT broker.")
except socket.gaierror as e:
    print(f"Connection error: {e}")
except Exception as e:
    print(f"Unexpected error: {e}")

# NLP to detect if teacher refers to student
def extract_student_id(message):
    match = re.search(r'\b(student_\d+)\b', message.lower())
    return match.group(1) if match else None

# Track activation
activated_student = None
emotion_history = deque(maxlen=15)

try:
    while True:
        teacher_input = input("Teacher says (or just press Enter to continue): ").strip()

        if teacher_input:
            student = extract_student_id(teacher_input)
            if student:
                activated_student = student
                print(f"[System] Now monitoring: {activated_student}")
            else:
                print("[System] No student name detected in message.")

        js_img = video_frame()
        if not js_img:
            continue

        frame = js_to_image(js_img)

        try:
            result = DeepFace.analyze(img_path=frame, actions=['emotion'], enforce_detection=False)
            emotion = result[0]['dominant_emotion']
        except Exception as e:
            print("Emotion detection error:", e)
            continue

        overlay_emotion_text(frame, emotion)
        clear_output(wait=True)
        display(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))

        if activated_student:
            # Append to emotion history
            emotion_history.append(emotion)

            inattentive_emotions = ['fear', 'sad', 'surprise']
            attentive_emotions = ['happy', 'neutral']

            inattentive_count = sum(1 for e in emotion_history if e in inattentive_emotions)

            if inattentive_count >= 5:
                message = f"{activated_student} is not active"
                print(message)
                client.publish(topic, message)
            else:
                # Don't keep sending "active" repeatedly
                if emotion in attentive_emotions and emotion_history.count(emotion) > 5:
                    message = f"{activated_student} is active"
                    print(message)
                    client.publish(topic, message)

        time.sleep(1)

except KeyboardInterrupt:
    print("Stopped by user")

finally:
    cv2.destroyAllWindows()
    print("Session ended.")


Stopped by user
Process finished.
